# LaunchML Week 4 Mini Capstone
## House Price Prediction with Linear Regression

**Student name:** Replace this text with your name.

This notebook is a guided template. Read every Markdown instruction, complete the missing work, and explain your decisions in your own words. Do not submit a notebook that contains only code.

**Project question:** How accurately can a Linear Regression model predict house sale prices from the available property features?

> The dataset is synthetic and educational. It is not official property-market data.


## 1. Problem definition

Before writing code, explain:

- What is the target variable?
- Is this regression or classification? Why?
- What information might help predict the target?

_Write your response in this cell._


## 2. Import libraries and configure the notebook

Use NumPy, Pandas, Matplotlib, and the required scikit-learn tools. Set a readable plotting style.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)


## 3. Load the dataset

Upload `launchml_house_prices.csv` to your Colab session if needed. If you are running this notebook from the supplied repository, use the path below.


In [2]:
DATA_PATH = "../data/launchml_house_prices.csv"
# In Google Colab, change DATA_PATH to the location of the uploaded CSV if necessary.
df = pd.read_csv(DATA_PATH)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/launchml_house_prices.csv'

## 4. Inspect the dataset

Run the checks below. Then write a short interpretation after the code. Do not only report numbers; explain what the checks tell you about the data.


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe(include="all").T


### Your inspection notes

Record the important observations from the previous output. Mention the target column, categorical columns, missing values, and duplicate rows.

_Write your response in this cell._


## 5. Clean and prepare the data

The modeling pipeline below handles missing numerical values with the median, missing categorical values with the most frequent category, and categorical values with one-hot encoding. The preprocessing is fitted only on the training data through the pipeline, which helps prevent information from the test set leaking into training.

Before using it, identify the numerical and categorical predictor columns. Keep `sale_price` out of `X`.


In [ ]:
target = "sale_price"
X = df.drop(columns=[target])
y = df[target]

numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## 6. Exploratory analysis

Use tables and charts to investigate the data before modeling. The first required visualization is a scatter plot of house area and sale price. Add a fitted line using `np.polyfit`.

Write an interpretation below the chart. A relationship does not prove causation.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["house_area_sqft"], df["sale_price"], alpha=0.55)
valid = df[["house_area_sqft", "sale_price"]].dropna()
line_x = np.linspace(valid["house_area_sqft"].min(), valid["house_area_sqft"].max(), 100)
line_y = np.polyval(np.polyfit(valid["house_area_sqft"], valid["sale_price"], 1), line_x)
plt.plot(line_x, line_y, color="crimson", linewidth=2, label="Fitted line")
plt.title("House Area and Sale Price")
plt.xlabel("House area (sq ft)")
plt.ylabel("Sale price")
plt.legend()
plt.tight_layout()
plt.savefig("visualization_1_house_area_vs_price.png", dpi=150)
plt.show()


### Interpretation of visualization 1

What pattern do you observe? Mention the direction, strength, and any unusual points you notice.

_Write your response in this cell._


## 7. Split the data and train Linear Regression

Use a reproducible 80/20 train-test split. The model must be Linear Regression only.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 8. Evaluate the model

Calculate the required metrics. Then explain what each value means in the context of price prediction.


In [3]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [mae, mse, rmse, r2]
})
metrics


NameError: name 'y_test' is not defined

### Metric interpretation

- **MAE:** Explain the average absolute prediction error in price units.
- **MSE:** Explain what squaring errors does to the metric.
- **RMSE:** Explain why it is often easier to interpret than MSE.
- **R²:** Explain what the value suggests about variation explained on the test set.

_Write your interpretation in this cell._


## 9. Review predictions

Create a table with actual prices, predicted prices, and errors. Then discuss whether the largest errors appear concentrated in particular price ranges.


In [ ]:
prediction_review = pd.DataFrame({
    "actual_sale_price": y_test.to_numpy(),
    "predicted_sale_price": y_pred
})
prediction_review["error"] = prediction_review["actual_sale_price"] - prediction_review["predicted_sale_price"]
prediction_review["absolute_error"] = prediction_review["error"].abs()
prediction_review.sort_values("absolute_error", ascending=False).head(10)


## 10. Required visualization 2: actual versus predicted prices

Points close to the reference line represent predictions close to actual values.


In [4]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
plt.plot([line_min, line_max], [line_min, line_max], color="crimson", linestyle="--", label="Perfect prediction")
plt.title("Actual versus Predicted Sale Prices")
plt.xlabel("Actual sale price")
plt.ylabel("Predicted sale price")
plt.legend()
plt.tight_layout()
plt.savefig("visualization_2_actual_vs_predicted.png", dpi=150)
plt.show()


NameError: name 'y_test' is not defined

<Figure size 700x600 with 0 Axes>

### Interpretation of visualization 2

Explain how closely the points follow the reference line and what that suggests about prediction accuracy.

_Write your response in this cell._


## 11. Required visualization 3: prediction errors

A residual is actual value minus predicted value. A random-looking pattern around zero is generally easier to interpret than a clear curve or funnel shape.


In [ ]:
residuals = y_test.to_numpy() - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color="crimson", linestyle="--")
plt.title("Residuals versus Predicted Sale Prices")
plt.xlabel("Predicted sale price")
plt.ylabel("Residual (actual - predicted)")
plt.tight_layout()
plt.savefig("visualization_3_residuals.png", dpi=150)
plt.show()


### Interpretation of visualization 3

Describe whether the residuals are centered around zero, whether their spread changes, and whether you see unusual points or patterns.

_Write your response in this cell._


## 12. Interpretation and limitations

Write a complete interpretation using your own metric values and visual evidence. Address:

1. Which features appeared most related to price?
2. What did you do with missing values and why?
3. How well did the model perform?
4. Where did it make larger errors?
5. What are at least three limitations?

Remember that the data are synthetic and that association does not prove causation.

_Write your response in this cell._


## 13. Final conclusion

Answer the project question directly in one or two paragraphs. State what you learned about data preparation, Linear Regression, evaluation, and communicating results.

_Write your response in this cell._


## 14. Submission reminder

Before submission:

- Run all cells from top to bottom.
- Confirm there are no errors.
- Move the three PNG files into your repository `images/` folder.
- Complete the written report.
- Upload the notebook, report, dataset, and images to GitHub.
- Check that your README explains how to run the project.
